# Reproduce GCN Baseline Results
This notebook loads the best model checkpoint and its configuration to regenerate metrics on the QM9 test set.

In [ ]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Model Definition

In [ ]:
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_layers, dropout=0.0):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        self.convs.append(GCNConv(in_channels, hidden_channels))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 1):
            self.convs.append(GCNConv(hidden_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))

        self.dropout = dropout
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels // 2),
            nn.ReLU(),
            nn.Linear(hidden_channels // 2, 1),
        )

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = global_mean_pool(x, batch)
        return self.head(x).squeeze(-1)

## 2. Load Configuration and Initialize Model

In [ ]:
config_path = 'gcn_config.json'
model_path  = 'gcn_best_model.pt'

if not os.path.exists(config_path) or not os.path.exists(model_path):
    raise FileNotFoundError('Required model or config file missing. Please run training first.')

with open(config_path, 'r') as f:
    config = json.load(f)

print(json.dumps(config, indent=2))

In [ ]:
# QM9 node features: 11 dimensions
in_channels = 11

model = GCN(
    in_channels=in_channels,
    hidden_channels=config['hidden_channels'],
    num_layers=config['num_layers'],
    dropout=config['dropout'],
).to(device)

model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()
print(f'Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

## 3. Load Dataset

In [ ]:
dataset     = QM9(root='../data/QM9')
test_dataset = dataset[120000:]
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False)

target_mean = config['target_mean']
target_std  = config['target_std']
TARGET_IDX  = config['target_idx']

print(f'Test set size: {len(test_dataset)}')

## 4. Evaluate and Report

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    total_mae = 0.0
    for batch in tqdm(loader, desc='Evaluating'):
        batch = batch.to(device)
        x     = batch.x.float()
        pred  = model(x, batch.edge_index, batch.batch) * target_std + target_mean
        total_mae += (pred - batch.y[:, TARGET_IDX]).abs().sum().item()
    return 1000 * total_mae / len(loader.dataset)  # eV -> meV

test_mae = evaluate(model, test_loader)
print(f'Reproduced Test MAE: {test_mae:.2f} meV')